In [ ]:
# Load env variables
import json
import re
import ast

from statistics import mean
from dotenv import load_dotenv
load_dotenv()

# Create an API client
from anthropic import Anthropic
from IPython.display import Markdown

client=Anthropic()
model= "claude-haiku-4-5"

def add_user_message(messages, text):
    user_message = {
        "role": "user",
        "content": text
    }
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {
        "role": "assistant",
        "content": text
    }
    messages.append(assistant_message)    

def chat(messages, system=None, stop_sequences=[]):
    params = {   
        "model": model,
        "max_tokens": 1000,
        "messages": messages
    }
    if system:
        params["system"] = system
    if stop_sequences:
        params["stop_sequences"] = stop_sequences

    response = client.messages.create(**params)
    return response.content[0].text


def generate_dataset():
    prompt= """ Generate an evaluation dataset for a prompt evaluation. the dataset will be used to evaluate prompts that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects, each representing task that requires Python, JSON, or a Regex to complete.

    Example output:
    ```json
    [ 
    {
    "task": "Description of task",
    "format": "json or python or regex"
    },
    ....additonal
    ]
    ```
* Focus on tasks that can be solved by writing a single Python function, a single JSON object or a single JSON regex
* Focus on tasks that do not require writing much code

Please generate 3 objects
"""

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages,"```json")
    text = chat(messages,stop_sequences=["```"])
    return json.loads(text)

dataset = generate_dataset()
print(dataset)

with open('dataset.json','w') as f:
    json.dump(dataset,f,indent=2)


def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
    Please solve the following task:
    {test_case["task"]}
    * Respond only with Python, JSON, or a plain Regex
    * Do not add any comments or commentary or explanation
"""

    messages = []
    add_user_message(messages,prompt)
    add_assistant_message(messages,"```code")
    output = chat(messages,stop_sequences=["```"])
    return output


def run_test_case(test_case):
    """Calls run_promt, then grades the result"""
    output = run_prompt(test_case)

    #Grade the output
    model_grade = grade_by_model(test_case,output)
    model_score = model_grade["score"]
    syntax_score = grade_syntax(output,test_case)
    reasoning = model_grade["reasoning"]
    score = (model_score + syntax_score) / 2

    return {
        "output" : output,
        "test_case" : test_case,
        "score": score,
        "reasoning": reasoning
    }

def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    average_score = mean([result["score"] for result in results]) 
    print(f"Average score: {average_score}")   
    return results


def grade_by_model(test_case,output):
    # Create evaluation prompt
    eval_prompt = f"""
    You are an expert code reviewer. Evaluate this AI-generated solution.
    Task: {test_case["task"]}
    Solution: {output}

    Provide your evaluation as a structred JSON object with:
    - "strengths": An array of 1-3 key strenghts
    - "weaknesses": An array of 1-3 key areas for improvement
    - "reasoning": A concise explanation of your assesment
    - "score": A number between 1-10
    """

    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")

    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)

def grade_syntax(output, test_case):
    format = test_case["format"]
    if format == "json":
        return validate_json(output)
    elif format == "python":
        return validate_python(output)
    else:
        return validate_regex(output)


def validate_json(text):
    try:
         json.loads(text.strip())
         return 10
    except json.JSONDecodeError:
         return 0

def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0


def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0 
        
    
with open("dataset.json","r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

print(json.dumps(results, indent=2))



[{'task': "Parse an AWS S3 bucket name and object key from an S3 URI in the format 's3://bucket-name/path/to/object'", 'format': 'regex'}, {'task': 'Create a JSON CloudFormation template snippet for an AWS Lambda function resource with basic properties (FunctionName, Runtime, Handler, Role)', 'format': 'json'}, {'task': "Write a Python function that takes an AWS IAM policy document as a dictionary and returns a list of all Action statements that contain wildcards ('*')", 'format': 'python'}]


NameError: name 'score' is not defined